In [ ]:

import os
import math
import requests
import json
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from urllib.parse import parse_qs, urlparse
import sys

API_BASE_URL = os.getenv('CASE_MGMT_BACKEND_URL', 'http://localhost:3090') + "/api/v1/jupyter/proxy/network-analysis/transaction/"
# Shared secret for proxy requests (set JUPYTER_SHARED_SECRET in environment)
JUPYTER_SHARED_SECRET = os.getenv('JUPYTER_SHARED_SECRET')
if JUPYTER_SHARED_SECRET:
    headers = {'x-jupyter-secret': JUPYTER_SHARED_SECRET}
else:
    headers = {}
# FALLBACK_ACCOUNT_ID = "cdtrAcct_8f37705af293453c992da08718fd8e9e"
query_string = os.environ.get("QUERY_STRING", "")
params = parse_qs(query_string)
account_id = params.get('accountId', [None])[0]

def fetch_transaction_network(account_id):
    url = f"{API_BASE_URL}{account_id}"
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except Exception:
       
        return None


In [ ]:

# account_id = get_account_id_from_url()
data = fetch_transaction_network(account_id)


In [ ]:
if not data:
    display(HTML(f"<div style='color:#ef4444'>No network data available for account: {account_id}</div>"))
else:
    nodes = []
    edges = []

    center = data.get('centerAccount')
    network_summary = center.get('networkSummary', {}) if center else {}
    
    if center:
        nodes.append({
            'id': center.get('accountId'),
            'label': center.get('accountHolder'),
            'type': 'center',
            'status': 'alert',
            'isCenter': True,
            'isUnderInvestigation': True,
            'networkSummary': network_summary
        })

    for acc in data.get('connectedAccounts', []):
        tx_stats = acc.get('transactionStats', {})
        nodes.append({
            'id': acc.get('accountId'),
            'label': acc.get('accountHolder'),
            'type': 'connected',
            'status': 'alert' if acc.get('hasAlert') else 'normal',
            'isCenter': False,
            'isUnderInvestigation': acc.get('isUnderInvestigation', False),
            'flowDirection': acc.get('flowDirection', ''),
            'totalTransactions': tx_stats.get('totalTransactions'),
            'totalValue': tx_stats.get('totalValue'),
            'averageValue': tx_stats.get('averageValue'),
            'velocity': tx_stats.get('velocity'),
            'firstTransactionDate': acc.get('firstTransactionDate'),
            'lastTransactionDate': acc.get('lastTransactionDate')
        })

    edges = data.get('edges', [])

    def get_node_size(node):
        if node.get('isCenter'):
            return 95
        if node.get('status') == 'alert':
            return 80
        return 75

    size_map = {n.get('id'): get_node_size(n) for n in nodes if n.get('id')}

    # Calculate positions
    positions = {}
    center_idx = 0

    for i, n in enumerate(nodes):
        if n.get('isCenter'):
            center_idx = i
            break

    radius = 280
    n_nodes = len(nodes)
    angle_step = 2 * math.pi / max(n_nodes - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1

    # Create edge traces with different styles
    solid_edge_x = []
    solid_edge_y = []
    dashed_edge_x = []
    dashed_edge_y = []

    center_id = center.get('accountId') if center else None

    def offset_edge(x0, y0, x1, y1, start_offset, end_offset):
        dx = x1 - x0
        dy = y1 - y0
        dist = math.hypot(dx, dy)
        if dist == 0:
            return x0, y0, x1, y1
        ux = dx / dist
        uy = dy / dist
        return (
            x0 + ux * start_offset,
            y0 + uy * start_offset,
            x1 - ux * end_offset,
            y1 - uy * end_offset
        )

    for e in edges:
        s = e.get('source')
        t = e.get('target')
        if s in positions and t in positions:
            x0, y0 = positions[s]
            x1, y1 = positions[t]

            start_offset = (size_map.get(s, 75) / 2) + 6
            end_offset = (size_map.get(t, 75) / 2) + 6
            x0, y0, x1, y1 = offset_edge(x0, y0, x1, y1, start_offset, end_offset)

            # Outbound from center = solid, Inbound to center = dashed
            if s == center_id:
                solid_edge_x += [x0, x1, None]
                solid_edge_y += [y0, y1, None]
            else:
                dashed_edge_x += [x0, x1, None]
                dashed_edge_y += [y0, y1, None]

    # Solid edges (outbound)
    solid_edge_trace = go.Scatter(
        x=solid_edge_x, y=solid_edge_y,
        mode='lines',
        line=dict(width=2, color='#ef4444', dash='solid'),
        hoverinfo='none',
        showlegend=False
    )

    # Dashed edges (inbound)
    dashed_edge_trace = go.Scatter(
        x=dashed_edge_x, y=dashed_edge_y,
        mode='lines',
        line=dict(width=2, color='#22c55e', dash='dash'),
        hoverinfo='none',
        showlegend=False
    )

    node_x = []
    node_y = []
    node_text = []
    node_labels = []
    marker_size = []
    marker_color = []
    marker_line_color = []
    marker_line_width = []
    node_ids = []

    ring_x = []
    ring_y = []
    ring_sizes = []

    for n in nodes:
        nid = n.get('id')
        if nid not in positions:
            continue
        x, y = positions[nid]
        node_x.append(x)
        node_y.append(y)
        label = n.get('label') or ''
        account_id_value = nid or '-'
        node_text.append(f"<b>{label}</b><br>Account ID: {account_id_value}")
        node_labels.append(label or 'Unknown')

        is_center = n.get('isCenter')
        status = n.get('status')
        is_under_investigation = n.get('isUnderInvestigation')

        if is_center:
            color = '#fee2e2'
            line_color = '#ef4444'
            size = 95
            line_width = 3
        elif status == 'alert':
            color = '#fee2e2'
            line_color = '#ef4444'
            size = 80
            line_width = 2
        else:
            color = '#f1f5f9'
            line_color = '#94a3b8'
            size = 75
            line_width = 2

        if is_center or is_under_investigation:
            ring_x.append(x)
            ring_y.append(y)
            ring_sizes.append(size + 12)

        marker_color.append(color)
        marker_line_color.append(line_color)
        marker_line_width.append(line_width)
        marker_size.append(size)
        node_ids.append(nid)

    ring_trace = go.Scatter(
        x=ring_x,
        y=ring_y,
        mode='markers',
        marker=dict(
            size=ring_sizes,
            color='rgba(0,0,0,0)',
            line=dict(width=3, color='#f59e0b')
        ),
        hoverinfo='none',
        showlegend=False
    )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers+text',
        text=node_labels,
        hovertext=node_text,
        hoverinfo='text',
        customdata=node_ids,
        marker=dict(
            showscale=False,
            color=marker_color,
            size=marker_size,
            line=dict(width=marker_line_width, color=marker_line_color)
        ),
        textposition='middle center',
        textfont=dict(size=10, color='#1e293b', family='Arial, sans-serif'),
        showlegend=False
    )

    # Use network summary from API
    connected_count = network_summary.get('connectedAccounts', 0)
    alert_count = network_summary.get('accountsWithAlerts', 0)
    outbound_count = network_summary.get('outboundConnections', 0)
    inbound_count = network_summary.get('inboundConnections', 0)

    annotations = []
    if center_id in positions:
        cx, cy = positions[center_id]
        annotations.append(
            dict(
                x=cx,
                y=cy - 22,
                text='⚠',
                showarrow=False,
                font=dict(size=18, color='#ef4444', family='Arial, sans-serif')
            )
        )

    fig = go.Figure(
        data=[dashed_edge_trace, solid_edge_trace, ring_trace, node_trace],
        layout=go.Layout(
            title=None,
            showlegend=False,
            hovermode='closest',
            dragmode=False,
            margin=dict(b=20, l=20, r=20, t=20),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-380, 380]),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-380, 380]),
            height=560,
            paper_bgcolor='#ffffff',
            plot_bgcolor='#ffffff',
            annotations=annotations
        )
    )

    graph_id = 'tx-network-graph'
    graph_html = pio.to_html(fig, full_html=False, config={'displayModeBar': False, 'scrollZoom': False}, div_id=graph_id)

    # Legend HTML
    legend_html = """
    <div style="position: absolute; bottom: 12px; left: 12px; background: white; padding: 10px; border-radius: 8px; border: 1px solid #e2e8f0; box-shadow: 0 4px 10px rgba(15,23,42,0.08); font-family: Arial, sans-serif; font-size: 10px;">
        <div style="font-weight: 600; margin-bottom: 8px; color: #0f172a; font-size: 11px;">Legend</div>
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <div style="width: 12px; height: 12px; border-radius: 50%; background: #fee2e2; border: 2px solid #ef4444; margin-right: 6px;"></div>
            <span style="color: #475569;">Alert Triggered</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <div style="width: 12px; height: 12px; border-radius: 50%; background: #f1f5f9; border: 2px solid #94a3b8; margin-right: 6px;"></div>
            <span style="color: #475569;">Normal Account</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 8px;">
            <div style="width: 12px; height: 12px; border-radius: 50%; background: #fee2e2; border: 2px solid #ef4444; margin-right: 6px; box-shadow: 0 0 0 2px #f59e0b;"></div>
            <span style="color: #475569;">Under Investigation</span>
        </div>
        <div style="border-top: 1px solid #e2e8f0; margin: 8px 0 6px 0;"></div>
        <div style="display: flex; align-items: center; margin-bottom: 4px;">
            <svg width="22" height="6" style="margin-right: 6px;">
                <line x1="0" y1="3" x2="16" y2="3" stroke="#ef4444" stroke-width="2" />
                <polygon points="16,1 22,3 16,5" fill="#ef4444" />
            </svg>
            <span style="color: #475569;">Outbound Flow</span>
        </div>
        <div style="display: flex; align-items: center;">
            <svg width="22" height="6" style="margin-right: 6px;">
                <line x1="0" y1="3" x2="16" y2="3" stroke="#22c55e" stroke-width="2" stroke-dasharray="4 3" />
                <polygon points="16,1 22,3 16,5" fill="#22c55e" />
            </svg>
            <span style="color: #475569;">Inbound Flow</span>
        </div>
    </div>
    """

    # Zoom Controls HTML
    zoom_controls_html = """
    <div style="position: absolute; top: 16px; right: 16px; background: white; padding: 8px; border-radius: 10px; border: 1px solid #e2e8f0; box-shadow: 0 4px 10px rgba(15,23,42,0.08); display: flex; flex-direction: column; gap: 8px;">
        <button id="zoom-in-btn" style="width: 34px; height: 34px; border: 1px solid #e2e8f0; background: #ffffff; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; color: #1e293b;" onmouseover="this.style.background='#f1f5f9'" onmouseout="this.style.background='#ffffff'">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <circle cx="11" cy="11" r="8"></circle>
                <line x1="11" y1="8" x2="11" y2="14"></line>
                <line x1="8" y1="11" x2="14" y2="11"></line>
                <line x1="21" y1="21" x2="16.65" y2="16.65"></line>
            </svg>
        </button>
        <button id="zoom-out-btn" style="width: 34px; height: 34px; border: 1px solid #e2e8f0; background: #ffffff; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; color: #1e293b;" onmouseover="this.style.background='#f1f5f9'" onmouseout="this.style.background='#ffffff'">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <circle cx="11" cy="11" r="8"></circle>
                <line x1="8" y1="11" x2="14" y2="11"></line>
                <line x1="21" y1="21" x2="16.65" y2="16.65"></line>
            </svg>
        </button>
        <button id="reset-zoom-btn" style="width: 34px; height: 34px; border: 1px solid #e2e8f0; background: #ffffff; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; color: #1e293b;" onmouseover="this.style.background='#f1f5f9'" onmouseout="this.style.background='#ffffff'" title="Reset Zoom">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <path d="M3 12a9 9 0 0 1 9-9 9.75 9.75 0 0 1 6.74 2.74L21 8"></path>
                <path d="M21 3v5h-5"></path>
                <path d="M21 12a9 9 0 0 1-9 9 9.75 9.75 0 0 1-6.74-2.74L3 16"></path>
                <path d="M3 21v-5h5"></path>
            </svg>
        </button>
    </div>
    """

    # Sidebar HTML
    sidebar_html = f"""
    <div id="sidebar-panel" style="width: 300px; background: white; padding: 24px; font-family: Arial, sans-serif;">
        <div style="margin-bottom: 18px;">
            <div id="sidebar-title" style="font-size: 18px; font-weight: 600; color: #0f172a; margin-bottom: 12px;">Center Account</div>
            <div style="margin-bottom: 12px;">
                <div style="font-size: 11px; color: #64748b; text-transform: uppercase; margin-bottom: 4px;">Account ID</div>
                <div id="sidebar-account-id" style="font-size: 14px; color: #0f172a; font-weight: 500;">{center.get('accountId', '-') if center else '-'}</div>
            </div>
            <div style="margin-bottom: 8px;">
                <div style="font-size: 11px; color: #64748b; text-transform: uppercase; margin-bottom: 4px;">Account Holder</div>
                <div id="sidebar-account-holder" style="font-size: 14px; color: #0f172a; font-weight: 600;">{center.get('accountHolder', '-') if center else '-'}</div>
            </div>
        </div>

        <div id="flow-direction-section" style="margin-bottom: 16px; display: none;">
            <div style="font-size: 11px; color: #64748b; text-transform: uppercase; margin-bottom: 6px;">Flow Direction</div>
            <div id="flow-direction-value" style="font-size: 13px; color: #0f172a; font-weight: 600; display: flex; align-items: center; gap: 6px;"></div>
        </div>

        <div style="height: 1px; background: #e2e8f0; margin: 16px 0 18px 0;"></div>

        <div id="network-summary" style="background: #ffffff; padding: 0; border-radius: 8px;">
            <div id="sidebar-summary-title" style="font-size: 12px; font-weight: 700; color: #0f172a; margin-bottom: 12px;">Network Summary</div>
            <div id="stat-connected" style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 13px;">Connected Accounts:</span>
                <span id="stat-connected-value" style="font-weight: 600; color: #0f172a;">{connected_count}</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 13px;">Outbound Connections:</span>
                <span id="stat-outbound-value" style="font-weight: 600; color: #0f172a;">{outbound_count}</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 13px;">Inbound Connections:</span>
                <span id="stat-inbound-value" style="font-weight: 600; color: #0f172a;">{inbound_count}</span>
            </div>
            <div id="stat-alerts" style="display: flex; justify-content: space-between;">
                <span style="color: #64748b; font-size: 13px;">Accounts with Alerts:</span>
                <span id="stat-alerts-value" style="font-weight: 600; color: #ef4444;">{alert_count}</span>
            </div>
        </div>

        <div id="transaction-stats" style="margin-top: 16px; display: none;">
            <div style="font-size: 12px; font-weight: 700; color: #0f172a; margin-bottom: 12px;">Transaction Statistics</div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 13px;">Total Transactions:</span>
                <span id="stat-total-tx" style="font-weight: 600; color: #0f172a;">-</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 13px;">Total Value:</span>
                <span id="stat-total-value" style="font-weight: 600; color: #0f172a;">-</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 13px;">Average Value:</span>
                <span id="stat-average-value" style="font-weight: 600; color: #0f172a;">-</span>
            </div>
            <div style="display: flex; justify-content: space-between;">
                <span style="color: #64748b; font-size: 13px;">Velocity:</span>
                <span id="stat-velocity" style="font-weight: 700; color: #ef4444;">-</span>
            </div>
        </div>

        <div id="alert-box" style="margin-top: 16px; padding: 10px 12px; border: 1px solid #fecaca; background: #fef2f2; color: #dc2626; border-radius: 8px; font-size: 12px; font-weight: 600; display: none;">
            ⚠ Alert triggered on this account
        </div>
    </div>
    """

    # Interactive script
    interactive_script = f"""
    <script>
    (function() {{
        const nodes = {json.dumps(nodes)};
        const edges = {json.dumps(edges)};
        const centerAcct = {json.dumps(center)};
        const originalPositions = {json.dumps(positions)};
        const graphDiv = document.getElementById('{graph_id}');

        let selectedNodeId = centerAcct ? centerAcct.accountId : null;
        let currentZoomLevel = 1.0;
        const baseRange = 380;
        const minZoom = 0.3;
        const maxZoom = 3.0;

        const numberFormatter = new Intl.NumberFormat();
        const currencyFormatter = new Intl.NumberFormat(undefined, {{ style: 'currency', currency: 'USD', maximumFractionDigits: 2 }});

        function formatNumber(value) {{
            if (value === null || value === undefined || value === '') return '-';
            const numeric = Number(value);
            return Number.isFinite(numeric) ? numberFormatter.format(numeric) : String(value);
        }}

        function formatCurrency(value) {{
            if (value === null || value === undefined || value === '') return '-';
            const numeric = Number(value);
            return Number.isFinite(numeric) ? currencyFormatter.format(numeric) : String(value);
        }}

        function getScaledPositions() {{
            const scaledPos = {{}};
            for (const [nodeId, pos] of Object.entries(originalPositions)) {{
                scaledPos[nodeId] = [pos[0] * currentZoomLevel, pos[1] * currentZoomLevel];
            }}
            return scaledPos;
        }}

        function rebuildGraph() {{
            const positions = getScaledPositions();
            const size_map = {{}};
            const base_sizes = {{}};
            
            // Calculate base and scaled sizes
            nodes.forEach(n => {{
                let baseSize;
                if (n.isCenter) baseSize = 95;
                else if (n.status === 'alert') baseSize = 80;
                else baseSize = 75;
                
                base_sizes[n.id] = baseSize;
                size_map[n.id] = baseSize * currentZoomLevel;
            }});

            // Rebuild edges
            const solid_edge_x = [];
            const solid_edge_y = [];
            const dashed_edge_x = [];
            const dashed_edge_y = [];
            const center_id = centerAcct ? centerAcct.accountId : null;

            function offsetEdge(x0, y0, x1, y1, start_offset, end_offset) {{
                const dx = x1 - x0;
                const dy = y1 - y0;
                const dist = Math.hypot(dx, dy);
                if (dist === 0) return [x0, y0, x1, y1];
                const ux = dx / dist;
                const uy = dy / dist;
                return [
                    x0 + ux * start_offset,
                    y0 + uy * start_offset,
                    x1 - ux * end_offset,
                    y1 - uy * end_offset
                ];
            }}

            edges.forEach(e => {{
                const s = e.source;
                const t = e.target;
                if (positions[s] && positions[t]) {{
                    let [x0, y0] = positions[s];
                    let [x1, y1] = positions[t];
                    
                    const start_offset = (size_map[s] || 75) / 2 + 6;
                    const end_offset = (size_map[t] || 75) / 2 + 6;
                    [x0, y0, x1, y1] = offsetEdge(x0, y0, x1, y1, start_offset, end_offset);

                    if (s === center_id) {{
                        solid_edge_x.push(x0, x1, null);
                        solid_edge_y.push(y0, y1, null);
                    }} else {{
                        dashed_edge_x.push(x0, x1, null);
                        dashed_edge_y.push(y0, y1, null);
                    }}
                }}
            }});

            // Rebuild nodes
            const node_x = [];
            const node_y = [];
            const ring_x = [];
            const ring_y = [];
            const node_sizes = [];
            const ring_sizes = [];
            const node_line_widths = [];

            nodes.forEach(n => {{
                const nid = n.id;
                if (positions[nid]) {{
                    const [x, y] = positions[nid];
                    node_x.push(x);
                    node_y.push(y);
                    node_sizes.push(size_map[nid]);
                    
                    // Scale line widths
                    let baseLineWidth = n.isCenter ? 3 : 2;
                    node_line_widths.push(baseLineWidth * currentZoomLevel);

                    if (n.isCenter || n.isUnderInvestigation) {{
                        ring_x.push(x);
                        ring_y.push(y);
                        ring_sizes.push(size_map[nid] + 12 * currentZoomLevel);
                    }}
                }}
            }});

            // Update all traces with positions and sizes
            Plotly.restyle(graphDiv, {{
                'x': [dashed_edge_x, solid_edge_x, ring_x, node_x],
                'y': [dashed_edge_y, solid_edge_y, ring_y, node_y],
                'marker.size': [undefined, undefined, ring_sizes, node_sizes],
                'line.width': [2 * currentZoomLevel, 2 * currentZoomLevel, undefined, undefined],
                'marker.line.width': [undefined, undefined, 3 * currentZoomLevel, node_line_widths],
                'textfont.size': [undefined, undefined, undefined, 10 * currentZoomLevel]
            }}, [0, 1, 2, 3]);

            // Update axis ranges to fit the zoomed graph
            const dynamicRange = baseRange * Math.max(1, currentZoomLevel);
            Plotly.relayout(graphDiv, {{
                'xaxis.range': [-dynamicRange, dynamicRange],
                'yaxis.range': [-dynamicRange, dynamicRange]
            }});

            // Update center alert annotation with scaled size
            if (center_id && positions[center_id]) {{
                const [cx, cy] = positions[center_id];
                const scaledOffset = 22 * currentZoomLevel;
                const scaledFontSize = 18 * currentZoomLevel;
                Plotly.relayout(graphDiv, {{
                    'annotations[0].x': cx,
                    'annotations[0].y': cy - scaledOffset,
                    'annotations[0].font.size': scaledFontSize
                }});
            }}
        }}

        function updateSidebar(nodeId) {{
            selectedNodeId = nodeId;
            const node = nodes.find(n => n.id === nodeId);
            if (!node) return;

            const title = node.isCenter ? 'Center Account' : 'Connected Account';
            document.getElementById('sidebar-title').innerText = title;
            document.getElementById('sidebar-account-id').innerText = node.id || '-';
            document.getElementById('sidebar-account-holder').innerText = node.label || '-';

            const flowSection = document.getElementById('flow-direction-section');
            const flowValue = document.getElementById('flow-direction-value');
            const summarySection = document.getElementById('network-summary');
            const txSection = document.getElementById('transaction-stats');
            const alertBox = document.getElementById('alert-box');

            if (node.isCenter) {{
                summarySection.style.display = 'block';
                txSection.style.display = 'none';
                flowSection.style.display = 'none';
                alertBox.style.display = 'none';

                document.getElementById('sidebar-summary-title').innerText = 'Network Summary';
                document.getElementById('stat-connected').style.display = 'flex';
                document.getElementById('stat-alerts').style.display = 'flex';

                const ns = node.networkSummary || {{}};
                document.getElementById('stat-connected-value').innerText = ns.connectedAccounts || 0;
                document.getElementById('stat-outbound-value').innerText = ns.outboundConnections || 0;
                document.getElementById('stat-inbound-value').innerText = ns.inboundConnections || 0;
                document.getElementById('stat-alerts-value').innerText = ns.accountsWithAlerts || 0;
            }} else {{
                summarySection.style.display = 'none';
                txSection.style.display = 'block';
                flowSection.style.display = 'block';
                alertBox.style.display = node.status === 'alert' ? 'block' : 'none';

                // Use flowDirection from API
                const flowDir = node.flowDirection || 'No direct flow';
                let flowIcon = '';
                if (flowDir.includes('Inbound')) {{
                    flowIcon = '←';
                }} else if (flowDir.includes('Outbound')) {{
                    flowIcon = '→';
                }} else if (flowDir.includes('Bidirectional')) {{
                    flowIcon = '↔';
                }}

                flowValue.innerHTML = '<span style="font-size:14px; color:#ef4444;">' + flowIcon + '</span><span>' + flowDir + '</span>';

                document.getElementById('stat-total-tx').innerText = formatNumber(node.totalTransactions);
                document.getElementById('stat-total-value').innerText = formatCurrency(node.totalValue);
                document.getElementById('stat-average-value').innerText = formatCurrency(node.averageValue);
                document.getElementById('stat-velocity').innerText = node.velocity || '-';
            }}
        }}

        function zoomIn() {{
            const newZoom = currentZoomLevel * 1.3;
            if (newZoom <= maxZoom) {{
                currentZoomLevel = newZoom;
                rebuildGraph();
                updateZoomButtons();
            }}
        }}

        function zoomOut() {{
            const newZoom = currentZoomLevel / 1.3;
            if (newZoom >= minZoom) {{
                currentZoomLevel = newZoom;
                rebuildGraph();
                updateZoomButtons();
            }}
        }}

        function resetZoom() {{
            currentZoomLevel = 1.0;
            rebuildGraph();
            updateZoomButtons();
        }}

        function updateZoomButtons() {{
            const zoomInBtn = document.getElementById('zoom-in-btn');
            const zoomOutBtn = document.getElementById('zoom-out-btn');
            const resetBtn = document.getElementById('reset-zoom-btn');
            
            if (currentZoomLevel * 1.3 > maxZoom) {{
                zoomInBtn.style.opacity = '0.4';
                zoomInBtn.style.cursor = 'not-allowed';
            }} else {{
                zoomInBtn.style.opacity = '1';
                zoomInBtn.style.cursor = 'pointer';
            }}
            
            if (currentZoomLevel / 1.3 < minZoom) {{
                zoomOutBtn.style.opacity = '0.4';
                zoomOutBtn.style.cursor = 'not-allowed';
            }} else {{
                zoomOutBtn.style.opacity = '1';
                zoomOutBtn.style.cursor = 'pointer';
            }}
            
            // Update reset button tooltip with current zoom
            const zoomPercent = Math.round(currentZoomLevel * 100);
            resetBtn.setAttribute('title', `Reset Zoom (current: ${{zoomPercent}}%)`);
        }}

        // Add mousewheel zoom support
        if (graphDiv) {{
            graphDiv.addEventListener('wheel', function(e) {{
                e.preventDefault();
                if (e.deltaY < 0) {{
                    zoomIn();
                }} else {{
                    zoomOut();
                }}
            }}, {{ passive: false }});
        }}

        if (graphDiv) {{
            graphDiv.on('plotly_click', d => {{
                if (d.points && d.points[0] && d.points[0].customdata) {{
                    const nodeId = d.points[0].customdata;
                    updateSidebar(nodeId);
                }}
            }});
        }}

        document.getElementById('zoom-in-btn').addEventListener('click', zoomIn);
        document.getElementById('zoom-out-btn').addEventListener('click', zoomOut);
        document.getElementById('reset-zoom-btn').addEventListener('click', resetZoom);
        
        // Initialize zoom button states
        updateZoomButtons();
    }})();
    </script>
    """

    container_html = f"""
    <div style="display: flex; background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px; overflow: hidden; box-shadow: 0 6px 16px rgba(15,23,42,0.08);">
        <div style="flex: 1; background: white; position: relative;">
            {graph_html}
            {legend_html}
            {zoom_controls_html}
        </div>
        <div style="width: 1px; background: #e2e8f0;"></div>
        {sidebar_html}
    </div>
    {interactive_script}
    """
    display(HTML(container_html))